<a href="https://colab.research.google.com/github/Kosuri069/IIT-Patna/blob/main/modelevaluation_regression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import r2_score

# -------------------------
# Dataset
# -------------------------
np.random.seed(0)
n = 120

data = {
    'age_years':   np.random.randint(1, 15, n),
    'km_driven':   np.random.randint(5000, 150000, n),
    'engine_cc':   np.random.randint(800, 3500, n),
    'fuel_type':   np.random.randint(0, 2, n),
    'noise_1':     np.random.randn(n),
    'noise_2':     np.random.randn(n),
}
df = pd.DataFrame(data)
df['price_lakhs'] = (
    -0.5  * df['age_years']
    - 0.00003 * df['km_driven']
    + 0.003 * df['engine_cc']
    + 2.0  * df['fuel_type']
    + np.random.randn(n) * 2
)

X = df.drop('price_lakhs', axis=1)
y = df['price_lakhs']

# -------------------------
# Task 1 — Baseline
# -------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

lr = LinearRegression()
lr.fit(X_train_scaled, y_train)

train_r2 = lr.score(X_train_scaled, y_train)
test_r2 = lr.score(X_test_scaled, y_test)

print("Task 1 — Linear Regression")
print(f"Train R²: {train_r2:.3f}")
print(f"Test R²:  {test_r2:.3f}")

# Comment:
# If train R² >> test R² → overfitting
# If both low → underfitting
# If similar and high → good fit


# -------------------------
# Task 2 — Ridge Tuning
# -------------------------
alphas = [0.01, 1, 10, 100, 500]
results = []

print("\nTask 2 — Ridge Tuning")

for alpha in alphas:
    ridge = Ridge(alpha=alpha)
    ridge.fit(X_train_scaled, y_train)

    train_r2 = ridge.score(X_train_scaled, y_train)
    test_r2 = ridge.score(X_test_scaled, y_test)

    results.append((alpha, test_r2))

    print(f"alpha={alpha:<6} Train R²={train_r2:.3f} Test R²={test_r2:.3f}")

best_alpha = sorted(results, key=lambda x: x[1], reverse=True)[0][0]
print(f"\nBest alpha: {best_alpha}")

# Comment:
# As alpha increases:
# - Bias more (model becomes simpler)
# - Variance less (less overfitting)
# Too large alpha → underfitting


# -------------------------
# Task 3 — Lasso Feature Selection
# -------------------------
lasso = Lasso(alpha=1.0, max_iter=10000)
lasso.fit(X_train_scaled, y_train)

print("\nTask 3 — Lasso Coefficients")

for feature, coef in zip(X.columns, lasso.coef_):
    print(f"{feature}: {coef:.4f}")

# Comment:
# Features with coefficient = 0 are removed by Lasso
# Likely 'noise_1' and 'noise_2' → irrelevant features eliminated


# -------------------------
# Task 4 — Validation Stability
# -------------------------
seeds = [42, 7, 123]

print("\nTask 4 — Stability Check")

for seed in seeds:
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=seed
    )

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    ridge = Ridge(alpha=best_alpha)
    ridge.fit(X_train_scaled, y_train)

    test_r2 = ridge.score(X_test_scaled, y_test)
    print(f"Seed {seed}: Test R² = {test_r2:.3f}")

# Comment:
# Small variation across seeds → model is stable
# Large variation → model is sensitive to data split (unstable)

Task 1 — Linear Regression
Train R²: 0.686
Test R²:  0.636

Task 2 — Ridge Tuning
alpha=0.01   Train R²=0.686 Test R²=0.637
alpha=1      Train R²=0.685 Test R²=0.641
alpha=10     Train R²=0.679 Test R²=0.664
alpha=100    Train R²=0.505 Test R²=0.555
alpha=500    Train R²=0.206 Test R²=0.233

Best alpha: 10

Task 3 — Lasso Coefficients
age_years: -0.6307
km_driven: -0.2007
engine_cc: 1.0973
fuel_type: 0.0000
noise_1: 0.0000
noise_2: 0.0000

Task 4 — Stability Check
Seed 42: Test R² = 0.664
Seed 7: Test R² = 0.665
Seed 123: Test R² = 0.603
